In [67]:
# =========================
#version07
# =========================

import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F

import matplotlib.pyplot as plt
from korean_lunar_calendar import KoreanLunarCalendar

from copy import deepcopy

from collections import defaultdict


plt.rcParams['font.family'] = 'AppleGothic'  # macOS
plt.rcParams['axes.unicode_minus'] = False

################# util #########################
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='min',
                 restore_best_weights=True, min_epochs=0,
                 relative=True, smooth_beta=0.0):
        """
        min_delta: 개선폭 기준 (relative=True면 비율, e.g., 0.005 = 0.5%)
        min_epochs: 이 에폭 전에는 중단 금지
        smooth_beta: 0~1, >0이면 EMA로 스무딩한 값을 모니터 (0이면 원시값)
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.restore_best_weights = restore_best_weights
        self.min_epochs = min_epochs
        self.relative = relative
        self.smooth_beta = smooth_beta

        self.best = None
        self.best_state = None
        self.wait = 0
        self.stop = False
        self.ema = None   # for smoothing

    def _improved(self, current):
        if self.best is None:
            return True
        # 상대/절대 개선폭 계산
        if self.mode == 'min':
            if self.relative:
                need = self.best * (1.0 - self.min_delta)
                return current < need
            else:
                return current < (self.best - self.min_delta)
        else:
            if self.relative:
                need = self.best * (1.0 + self.min_delta)
                return current > need
            else:
                return current > (self.best + self.min_delta)

    def step(self, current, model, epoch_idx: int):
        # 선택: 스무딩
        if self.smooth_beta > 0.0:
            self.ema = current if self.ema is None else (self.smooth_beta*self.ema + (1-self.smooth_beta)*current)
            monitor_val = self.ema
        else:
            monitor_val = current

        if self.best is None or self._improved(monitor_val):
            self.best = monitor_val
            self.best_state = deepcopy(model.state_dict())
            self.wait = 0
            return False

        # 아직 최소 에폭 전이면 기다림만 증가
        self.wait += 1
        if epoch_idx+1 < self.min_epochs:
            return False

        if self.wait >= self.patience:
            self.stop = True
            if self.restore_best_weights and self.best_state is not None:
                model.load_state_dict(self.best_state)
            return True
        return False

##########################################
#Loss
class SmapeLoss(nn.Module):
    def __init__(self, eps=1e-3): super().__init__(); self.eps=eps
    def forward(self, yhat, y):
        num = (yhat-y).abs()
        den = (yhat.abs()+y.abs()).clamp_min(self.eps)
        return (2.0*num/den).mean()

# 기존 weighted_huber_smape 교체
def weighted_huber_smape(
    yhat, y, w=None, *, delta=0.05, eps=1e-3, alpha=0.5
):
    hub = F.huber_loss(yhat, y, delta=delta, reduction='none')
    den = yhat.abs() + y.abs()
    if isinstance(eps, float) or isinstance(eps, int):
        den = torch.clamp(den, min=float(eps))
    else:
        den = torch.maximum(den, eps)
    smp = 2.0 * (yhat - y).abs() / den
    loss = alpha * hub + (1 - alpha) * smp
    if w is not None: loss = loss * w
    return loss.mean()

def smape_loss(pred, target, eps=1e-3, reduction='mean', ignore_zero_target=False):
    """
    ignore_zero_target=True이면 target==0인 위치는 sMAPE 평균에서 제외.
    (대회 채점 규칙과 정렬)
    """
    num = (pred - target).abs()
    den = (pred.abs() + target.abs()).clamp_min(eps)
    sm = 2.0 * num / den  # (B, PREDICT)

    if reduction == 'none' and not ignore_zero_target:
        return sm

    if ignore_zero_target:
        mask = (target != 0).float()                     # (B,P)
        # 각 샘플(행)별 유효 포인트 평균 → 배치 평균
        denom = mask.sum(dim=1).clamp_min(1.0)           # (B,)
        row_mean = (sm * mask).sum(dim=1) / denom        # (B,)
        return row_mean.mean() if reduction != 'none' else (sm, mask)

    # 기존 동작
    return sm.mean()

mse = nn.MSELoss()
ALPHA = 0.7      # 시작 0.3~0.5, 안정하면 0.6~0.7로 ↑
EPS   = 1e-3

def combined_loss(pred, target, alpha=ALPHA, eps=EPS):
    # alpha: sMAPE 비중 (0.3~0.7 범위 추천)
    l_mse = mse(pred, target)
    l_smape = smape_loss(pred, target, eps=eps)
    return (1-alpha) * l_mse + alpha * l_smape

######################################################

def adamw_params(model, wd=1e-4):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: 
            continue
        if p.ndim == 1 or n.endswith("bias") or "norm" in n.lower() or "embedding" in n.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {"params": decay, "weight_decay": wd},
        {"params": no_decay, "weight_decay": 0.0},
    ]
def add_ts_stats(
    df: pd.DataFrame,
    target_col: str = "clipped_SQ",   # 스케일 전(or 스케일 후) 타깃 중 택1
    date_col: str = "영업일자",
    lags = (1, 7, 14, 28),
    roll_windows = (7, 14, 28),
    ewm_spans = (7, 14),
    out_prefix: str = "",
    eps: float = 1e-3
) -> pd.DataFrame:
    """
    시계열 통계 피처 생성 (모두 과거만 사용)
    생성: lag_k, roll_mean_k, roll_std_k, ewm_mean_s, momentum_k, rel_level_k, vol_k
    - momentum_k  = (x_t - x_{t-k}) / (|x_{t-k}|+eps)
    - rel_level_k = x_t / (roll_mean_k + eps)
    - vol_k       = roll_std_k / (roll_mean_k + eps)
    """
    # 정렬 보장
    df = df.sort_values(date_col)
    x = df[target_col].astype("float32")

    # Lags
    for k in lags:
        df[f"{out_prefix}lag_{k}"] = x.shift(k).astype("float32")

    # Rolling mean/std (과거 window, 현재 포함 → 누수 방지 위해 shift(1) 후 rolling도 가능)
    for w in roll_windows:
        base = x.shift(1)  # 현재값 제외
        df[f"{out_prefix}roll_mean_{w}"] = base.rolling(w, min_periods=1).mean().astype("float32")
        df[f"{out_prefix}roll_std_{w}"]  = base.rolling(w, min_periods=1).std().fillna(0).astype("float32")

    # EWMA
    for s in ewm_spans:
        df[f"{out_prefix}ewm_mean_{s}"] = x.ewm(span=s, adjust=False).mean().astype("float32")

    # Momentum & Relative level & Volatility (대표 window=7 사용; 필요시 반복문 확장)
    for k in lags:
        df[f"{out_prefix}momentum_{k}"] = ((x - x.shift(k)) / (np.abs(x.shift(k)) + eps)).astype("float32")
    for w in roll_windows:
        m = df[f"{out_prefix}roll_mean_{w}"]
        s = df[f"{out_prefix}roll_std_{w}"]
        df[f"{out_prefix}rel_level_{w}"] = (x / (m + eps)).astype("float32")      # 수준/평균
        df[f"{out_prefix}vol_{w}"]       = (s / (m + eps)).astype("float32")      # 변동성/평균(무단위)

    return df

#Fixed Random Seed  & Setting Hyperparameter
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)

LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass
MONTH_SCALE = 12
PATIENCE = 10
MIN_DELTA = 0.0

MIN_SEQUENCE_COUNT = 10

# 키: '영업장명_메뉴명', 값: 'YYYY-MM-DD' (ISO 문자열 또는 datetime)
DISCONTINUED = {
    '담하_꼬막_비빔밥': '2024-04-01',
    '담하_들깨_양지탕': '2024-04-01',
    # ...
}

# === 공통 피처 정의 (학습/추론 동일) ===
FEATURES = [
    'clipped_SQ','rolling_mean_7','delta_scaled',
    # 월/연 Fourier (k=1..3)
    'month_sin1','month_cos1','month_sin2','month_cos2','month_sin3','month_cos3',
    'doy_sin1','doy_cos1','doy_sin2','doy_cos2','doy_sin3','doy_cos3',
    'holiday_prox','is_holiday',
    'w_sin1','w_sin2','w_cos1','w_cos2',
    'lag_7','lag_14','lag_28',
    'rel_level_7','rel_level_14',
    'vol_7','vol_14',
    'momentum_7','momentum_14',
    'ewm_mean_7','ewm_mean_14',
    'holiday_prox_lag1', 'holiday_prox_lag2', 'holiday_prox_lag3',
    'holiday_prox_lead1','holiday_prox_lead2','holiday_prox_lead3',
    'holiday_prox_lead4','holiday_prox_lead5','holiday_prox_lead6','holiday_prox_lead7'
]
FEAT = {name: i for i, name in enumerate(FEATURES)}
for k in ['rolling_mean_7','holiday_prox','momentum_7','vol_14','ewm_mean_7']:
    assert k in FEAT, f"Missing feature index: {k}"
    
def build_features(
    df: pd.DataFrame,
    scaler_xy: MinMaxScaler,
    scaler_delta: MinMaxScaler,
    fit: bool = False,
    date_col: str = '영업일자'
) -> pd.DataFrame:
    """
    학습/추론에서 동일하게 쓰는 피처 빌더.
    - clipped_SQ, rolling_mean_7 scaling
    - delta_scaled scaling
    - Fourier/holiday proximity/weekly sincos
    - ts-stats (lag/roll/ewm/momentum/rel_level/vol) + 결측 처리
    """
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # 요일/월/시즌
    out['weekday'] = out[date_col].dt.dayofweek.astype(int)
    m = out[date_col].dt.month.astype(np.int16)
    out['season'] = m.map({12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}).astype(int)

    # 휴일, Fourier, 주간 주기
    out = generate_combined_holiday_list(out, solar_md_holidays, lunar_solar_dates)
    out = add_fourier_seasonal_features(out, date_col=date_col)

    t = (out[date_col] - out[date_col].min()).dt.days.values
    for k in (1, 2):
        out[f'w_sin{k}'] = np.sin(2*np.pi*k*t/7).astype('float32')
        out[f'w_cos{k}'] = np.cos(2*np.pi*k*t/7).astype('float32')

    # holiday proximity (+ lag/lead)
    out = add_holiday_proximity(out, date_col, 'is_holiday', 'holiday_prox', K=10, return_what='prox')
    # 먼저 기존 lag/lead 컬럼이 있으면 정리(덮어쓰기 혼선 방지)
    _drop_cols = [f'holiday_prox_lag{k}' for k in range(1, PREDICT+1)] + \
                [f'holiday_prox_lead{k}' for k in range(1, PREDICT+1)]
    exist_drop = [c for c in _drop_cols if c in out.columns]
    if exist_drop:
        out.drop(columns=exist_drop, inplace=True)

    # 벡터화로 한 번에 생성 (1..PREDICT 모두)
    lag_cols  = {f'holiday_prox_lag{k}':  out['holiday_prox'].shift(k)   for k in range(1, PREDICT+1)}
    lead_cols = {f'holiday_prox_lead{k}': out['holiday_prox'].shift(-k)  for k in range(1, PREDICT+1)}
    out = out.assign(**lag_cols, **lead_cols)

    # 결측/타입 정리
    mk_cols = [f'holiday_prox_lag{k}' for k in range(1, PREDICT+1)] + \
            [f'holiday_prox_lead{k}' for k in range(1, PREDICT+1)]
    out[mk_cols] = out[mk_cols].fillna(0.0).astype('float32')
    
    # IQR clip / delta / rolling
    out['clipped_SQ']     = clip_iqr(out['매출수량']) if '매출수량' in out.columns else out['clipped_SQ']
    out['delta']          = out['clipped_SQ'].diff().fillna(0)
    out['rolling_mean_7'] = out['clipped_SQ'].rolling(window=7, min_periods=1).mean()

    # scaling
    if fit:
        scaler_xy.fit(out[['clipped_SQ', 'rolling_mean_7']])
        scaler_delta.fit(out[['delta']])
    out[['clipped_SQ', 'rolling_mean_7']] = scaler_xy.transform(out[['clipped_SQ','rolling_mean_7']])
    out[['delta_scaled']] = scaler_delta.transform(out[['delta']])

    # ts-stats (누수 안전 옵션: 현재값 제외하려면 shift(1) 사용)
    # 엄격 모드 예시:
    # out_ts = add_ts_stats(out.copy(), target_col="clipped_SQ", date_col=date_col, ...)
    # 부분만 교체하려면 add_ts_stats 내부에서 rolling을 x.shift(1).rolling(...)로 바꾸세요.
    out = add_ts_stats(out, target_col="clipped_SQ", date_col=date_col,
                       lags=(1,7,14,28), roll_windows=(7,14,28), ewm_spans=(7,14))

    # ✅ 모든 시계열 통계 파생 컬럼에서 NaN/Inf 제거
    ts_cols = [c for c in out.columns if c.startswith((
        'lag_', 'momentum_', 'roll_mean_', 'roll_std_', 'rel_level_', 'vol_', 'ewm_mean_'
    ))]
    out[ts_cols] = (out[ts_cols]
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0)
                    .astype('float32'))

    return out


def get_lunar_to_solar(years, lunar_month, lunar_day, span=1):
    calendar = KoreanLunarCalendar()
    dates = []
    for year in years:
        for offset in range(-span, span+1):
            try:
                calendar.setLunar(year, lunar_month, lunar_day + offset, False)
                dates.append(calendar.SolarIsoFormat())
            except:
                pass  # 예외 처리: 음력 마지막날 초과
    return dates
# 예시: 2023 ~ 2025
years = [2023, 2024, 2025]
lunar_solar_dates = []
lunar_solar_dates += get_lunar_to_solar(years, 1, 1, span=1)   # 설날 ±1
lunar_solar_dates += get_lunar_to_solar(years, 8, 15, span=1)  # 추석 ±1

solar_md_holidays = [
    (1, 1),   # 신정
    (3, 1),   # 삼일절
    (5, 5),   # 어린이날
    (6, 6),   # 현충일
    (8, 15),  # 광복절
    (10, 3),  # 개천절
    (10, 9),  # 한글날
    (12, 25), # 크리스마스
]

def generate_combined_holiday_list(df, solar_md_list, lunar_solar_list):
    df = df.copy()
    df['영업일자'] = pd.to_datetime(df['영업일자'])

    # 양력 기반 holiday 판별
    df['is_solar_holiday'] = df['영업일자'].apply(
        lambda x: (x.month, x.day) in solar_md_list
    )

    # 음력 변환된 holiday 포함
    lunar_set = set(pd.to_datetime(lunar_solar_list))
    df['is_lunar_holiday'] = df['영업일자'].isin(lunar_set)

    # 최종 통합
    df['is_holiday'] = (df['is_solar_holiday'] | df['is_lunar_holiday']).astype(int)
    df = df.drop(columns=['is_solar_holiday', 'is_lunar_holiday'])
    return df


def remove_leading_zeros_before_sales(df, min_zero_days=90):
    """
    매출 시작 전 연속 0이 일정 기간 이상이면, 그 전 구간 제거
    (단일 메뉴-업장 그룹 DataFrame을 가정)
    """
    sales_started = df['매출수량'] > 0
    if not sales_started.any():
        return df  # 매출이 전혀 없는 경우 그대로 반환

    first_sale_idx = sales_started.idxmax()

    # 매출 시작 전 구간이 충분히 긴 0으로 구성되어 있다면 제거
    df_before = df.loc[:first_sale_idx - 1]
    if len(df_before) >= min_zero_days and (df_before['매출수량'] == 0).all():
        return df.loc[first_sale_idx:]  # 매출 시작부터 반환
    else:
        return df  # 그대로 반환


def _extract_store_name(g: pd.DataFrame) -> str:
    """
    그룹 g에서 업장명 추출:
    - '영업장명' 컬럼이 있으면 그 값을 사용
    - 없으면 '영업장명_메뉴명'에서 첫 '_' 앞을 업장명으로 간주
    """
    if '영업장명' in g.columns:
        return str(g['영업장명'].iloc[0])
    # '영업장명_메뉴명'이 "업장명_메뉴명" 형태라고 가정
    full = str(g['영업장명_메뉴명'].iloc[0])
    return full.split('_', 1)[0]  # '_'가 여러 개여도 첫 구분만 사용

def filter_all_menus_by_leading_zeros(
    train_df: pd.DataFrame,
    min_zero_days: int = 90,
    apply_to_stores: list[str] | None = None,
    exclude_stores: list[str] | None = None,
    group_col: str = '영업장명_메뉴명',
) -> pd.DataFrame:
    """
    모든 메뉴-업장 그룹에 대해 remove_leading_zeros_before_sales를 적용하되,
    특정 업장에만(또는 특정 업장은 제외하고) 적용할 수 있도록 확장.

    Parameters
    ----------
    train_df : 전체 데이터프레임
    min_zero_days : 매출 시작 전 연속 0 최소 일수
    apply_to_stores : 적용 대상 업장명 리스트 (None이면 전 업장 대상)
    exclude_stores : 적용 제외 업장명 리스트 (None이면 제외 없음)
    group_col : 그룹화 기준 컬럼명 (기본: '영업장명_메뉴명')
    """
    parts = []
    apply_set   = set(apply_to_stores) if apply_to_stores is not None else None
    exclude_set = set(exclude_stores)  if exclude_stores  is not None else set()

    # 기존 순서 보존 원하면 sort=False 유지
    for _, g in train_df.groupby(group_col, sort=False):
        store = _extract_store_name(g)

        # 적용 여부 결정
        apply_flag = True
        if apply_set is not None:
            apply_flag = (store in apply_set)
        if store in exclude_set:
            apply_flag = False

        if apply_flag:
            parts.append(remove_leading_zeros_before_sales(g, min_zero_days))
        else:
            parts.append(g)

    if parts:
        return pd.concat(parts, ignore_index=True)
    return train_df.reset_index(drop=True)

# === 추가: Fourier 계절 피처 함수 ===
def add_fourier_seasonal_features(df: pd.DataFrame, date_col: str = '영업일자') -> pd.DataFrame:
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    m = df[date_col].dt.month.astype(np.int16)          # 1..12
    doy = df[date_col].dt.dayofyear.astype(np.int16)    # 1..365 (윤년은 무시해도 충분)
    # (A) 월 주기: 12개월 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'month_sin{k}'] = np.sin(2*np.pi*k*m/12).astype('float32')
        df[f'month_cos{k}'] = np.cos(2*np.pi*k*m/12).astype('float32')
    # (B) 연간 주기: 365일 주기, k=1..3 고차 조화항
    for k in (1, 2, 3):
        df[f'doy_sin{k}'] = np.sin(2*np.pi*k*doy/365).astype('float32')
        df[f'doy_cos{k}'] = np.cos(2*np.pi*k*doy/365).astype('float32')
    # (C) 범주형 월 인덱스(임베딩용)
    df['month_idx'] = (m - 1).astype(int)  # 0~11
    return df

def add_holiday_proximity(
    df: pd.DataFrame,
    date_col: str = '영업일자',
    holiday_col: str = 'is_holiday',
    out_col: str = 'holiday_prox',
    K: int = 7,
    return_what: str = 'prox',  # 'prox' 또는 'dist'
) -> pd.DataFrame:
    """
    캘린더 휴일 기준으로 각 날짜가 휴일에 얼마나 근접했는지 계산합니다.
    - prox: (K - min(dist_prev, dist_next)) / K ∈ [0,1], 당일 휴일=1, K일 이상 떨어지면 0
    - dist: min(dist_prev, dist_next) ∈ [0, K] (K로 클리핑)

    Notes
    -----
    * holiday_col은 미래를 '알 수 있는' 캘린더 정보이므로 누수 위험이 없습니다.
    * df의 원래 행 순서를 유지합니다.
    """
    if date_col not in df.columns:
        raise KeyError(f"'{date_col}' not in df")
    if holiday_col not in df.columns:
        raise KeyError(f"'{holiday_col}' not in df")

    # 원래 인덱스 저장
    orig_index = df.index

    # 날짜 정렬본으로 계산
    tmp = df[[date_col, holiday_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    tmp = tmp.sort_values(date_col)

    mask = tmp[holiday_col].astype(bool)
    # 휴일이면 그 날짜, 아니면 NaT
    s_h = tmp[date_col].where(mask)

    # 과거/미래 휴일 날짜
    prev_h = s_h.ffill()
    next_h = s_h.bfill()

    # 거리 계산(일수)
    dist_prev = (tmp[date_col] - prev_h).dt.days.astype('float32')
    dist_next = (next_h - tmp[date_col]).dt.days.astype('float32')

    # 휴일이 아예 없을 때 NaN → K+1로 대체
    dist_prev = dist_prev.fillna(K + 1)
    dist_next = dist_next.fillna(K + 1)

    # 최소 거리 후 K로 클리핑
    dist_h = np.minimum(dist_prev, dist_next).clip(0, K).astype('float32')

    if return_what == 'dist':
        out = dist_h
    elif return_what == 'prox':
        # 근접도: 0(멀다) ~ 1(당일 휴일)
        out = ((K - dist_h) / K).astype('float32')
    else:
        raise ValueError("return_what must be 'prox' or 'dist'")

    # 정렬 전 순서로 복원
    out = out.reindex(tmp.index)                # 안전: 이미 tmp와 동일
    out_df = pd.DataFrame({out_col: out}, index=tmp.index)
    out_df = out_df.reindex(orig_index)         # 원래 df 순서로

    # 원본 df에 컬럼으로 추가
    df[out_col] = out_df[out_col].values.astype('float32')
    return df

def ensure_time_major(x: torch.Tensor, lookback: int, n_features: int) -> torch.Tensor:
    """
    x shape을 (B, T, F)로 강제 정렬.
    - 올바르면 그대로 반환
    - (B, F, T)로 뒤집혀 있으면 permute(0,2,1)
    """
    if x.dim() != 3:
        raise ValueError(f"Expected 3D tensor, got {x.dim()}D: {tuple(x.shape)}")
    B, A, B2 = x.shape
    # 정상: (B, T, F)
    if A == lookback and B2 == n_features:
        return x
    # 뒤집힘: (B, F, T)
    if A == n_features and B2 == lookback:
        return x.permute(0, 2, 1).contiguous()
    # 그 외: 명시 에러로 빨리 잡기
    raise ValueError(f"Unexpected shape for sequence tensor: {tuple(x.shape)} (T={lookback}, F={n_features})")

#Define Model
class MultiEmbeddingLSTM(nn.Module):
    def __init__(
        self,
        input_dim,                  # 수치 feature 수
        hidden_dim=96,
        num_layers=2,
        output_dim=7,
        num_weekdays=7,
        weekday_embed_dim=3,
        num_seasons=4,
        season_embed_dim=2,
        num_months=12, month_embed_dim=2, use_month=False,
        dropout=0.3,
        proj_dim=32,               # ★ 투영 차원(없애려면 None)
        use_residual=False,
        # ★ 추가 옵션
        pool_mode: str = "last",   # "last" | "avg_last_k"
        pool_k: int = 14,                # 최근 k 스텝 평균
        head_mode: str = "mul",          # "mul"(배율) | "plain"(기존 FC)
        r_scale_base: float = 0.4,       # 기본 r_scale
        # ★ 피처 인덱스 (FEATURES 기준)
        rm7_idx: int = 1,                # rolling_mean_7
        prox_idx: int = 15,              # holiday_prox
        mom7_idx: int = 28,              # momentum_7
        vol14_idx: int = 27,
        use_horizon_embed: bool = True,
        horizon_embed_dim: int = 8                    
    ):
        super().__init__()
        
        self.output_dim = output_dim
        self.use_month = use_month
        
        self.pool_mode = pool_mode
        self.pool_k = pool_k
        self.head_mode = head_mode
        self.use_horizon_embed = use_horizon_embed
        self.r_scale_base = r_scale_base

        self.rm7_idx = rm7_idx
        self.prox_idx = prox_idx
        self.mom7_idx = mom7_idx
        self.vol14_idx = vol14_idx


        # 임베딩
        self.weekday_embedding = nn.Embedding(num_weekdays, weekday_embed_dim)
        self.season_embedding = nn.Embedding(num_seasons, season_embed_dim)

        # LayerNorm for embedding
        self.weekday_norm = nn.LayerNorm(weekday_embed_dim)
        self.season_norm = nn.LayerNorm(season_embed_dim)

        total_in = input_dim + weekday_embed_dim + season_embed_dim 

        if use_month:
            self.month_embedding = nn.Embedding(num_months, month_embed_dim)
            self.month_norm = nn.LayerNorm(month_embed_dim)
            total_in += month_embed_dim

        # Dropout after embedding
        self.embedding_dropout = nn.Dropout(dropout)

        # ---- 투영층 ----
        self.proj_dim = proj_dim
        self.use_residual = use_residual and (proj_dim is not None) and (proj_dim == total_in)  # 동일차원일 때만 유효

        if proj_dim is not None:
            self.proj = nn.Sequential(
                nn.Linear(total_in, proj_dim),
                nn.GELU(),
                nn.LayerNorm(proj_dim),
                nn.Dropout(dropout)
            )
            lstm_in = proj_dim
        else:
            self.proj = None
            lstm_in = total_in

        # LSTM
        self.lstm = nn.LSTM(lstm_in, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        # LSTM output dropout
        self.post_lstm_dropout = nn.Dropout(dropout)

        # ---- 헤드
        self.fc_plain = nn.Linear(hidden_dim, output_dim)     # plain 모드용
        # 변경(예: 8차원 horizon 임베딩 + 작은 MLP)
        self.h_emb = nn.Embedding(output_dim, 8)
        self.fc_ratio = nn.Sequential(
            nn.Linear(hidden_dim + 8, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, output_dim)
        )

        # ★ 추가: Horizon Embedding + MLP head
        if self.use_horizon_embed:
            self.h_emb = nn.Embedding(output_dim, horizon_embed_dim)
            self.head_mlp_ratio = nn.Sequential(
                nn.Linear(hidden_dim + horizon_embed_dim, hidden_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1)  # → 각 horizon마다 1개 ratio
            )
            self.head_mlp_plain = nn.Sequential(
                nn.Linear(hidden_dim + horizon_embed_dim, hidden_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim // 2, 1)  # → 각 horizon마다 1개 값
            )

    def _pool(self, hs: torch.Tensor) -> torch.Tensor:
        # hs: (B,T,H)
        if self.pool_mode == "avg_last_k":
            k = min(self.pool_k, hs.size(1))
            return hs[:, -k:, :].mean(dim=1)  # (B,H)
        else:
            return hs[:, -1, :]               # (B,H)
        
    def forward(self, x, weekday_ids, season_ids, month_ids=None):
        """
        x: (B, T, input_dim)
        weekday_ids: (B, T)
        season_ids: (B, T)
        """
        wd = self.weekday_norm(self.weekday_embedding(weekday_ids))  # (B,T,Dw)
        ss = self.season_norm(self.season_embedding(season_ids))     # (B,T,Ds)
        feats = [x, wd, ss]
        if self.use_month:
            mm = self.month_norm(self.month_embedding(month_ids))
            feats.append(mm)

        z = torch.cat(feats, dim=-1)                           # (B,T,total_in)
        z = self.embedding_dropout(z)

        if self.proj is not None:
            z_proj = self.proj(z)                                     # (B,T,proj_dim)
            if self.use_residual:
                z = z + z_proj                                        # 동일 차원일 때만
            else:
                z = z_proj

        hs, _ = self.lstm(z)                 # (B,T,H)
        hs = self.post_lstm_dropout(hs)
        ctx = self._pool(hs)                 # (B,H)

        B = x.size(0)

        if self.use_horizon_embed:
            # 0..P-1 horizon index 만들고 배치에 확장
            h_idx = torch.arange(self.output_dim, device=x.device).unsqueeze(0).expand(B, -1)  # (B,P)
            h_e   = self.h_emb(h_idx)  # (B,P,E)

            # ctx를 horizon 차원으로 확장
            ctx_exp = ctx.unsqueeze(1).expand(B, self.output_dim, ctx.size(1))  # (B,P,H)

            # concat 후 MLP
            ratio = torch.tanh(
                self.head_mlp_ratio(torch.cat([ctx_exp, h_e], dim=-1))  # (B,P,1)
            ).squeeze(-1)  # (B,P)
        else:
            # horizon 임베딩을 안 쓸 때는 fc_ratio를 hidden_dim만 받도록 정의하거나,
            # 임베딩 평균을 붙여 차원 맞춤(둘 중 택1). 아래는 "평균 임베딩" 방식:
            hmean = self.h_emb.weight.mean(dim=0, keepdim=True).expand(B, -1)  # (B,8)
            ratio = torch.tanh(self.fc_ratio(torch.cat([ctx, hmean], dim=-1)))  # (B,P)

        # --- 안전 가드: ratio를 (B, PREDICT)로 강제 ---
        if ratio.dim() == 3:
            ratio = ratio.squeeze(-1)
        if ratio.dim() != 2 or ratio.size(1) != self.output_dim:
            raise RuntimeError(f"[shape guard] ratio shape={tuple(ratio.shape)}, expected (B,{self.output_dim})")

        # 동적 r_scale
        prox = x[:, -1, self.prox_idx]          # (B,)
        wknd = (weekday_ids[:, -1] >= 5).float()
        mom7 = x[:, -1, self.mom7_idx]
        vol14= x[:, -1, self.vol14_idx]

        boost = self.r_scale_base + 0.4*prox + 0.1*wknd + 0.2*F.relu(mom7) + 0.1*F.relu(vol14 - 0.3)
        boost = boost.clamp(0.3, 0.9).unsqueeze(1)  # (B,1)

        # --- base를 (B, PREDICT)로 확장 (expand_as 금지) ---
        B = x.size(0)
        base = x[:, -1, self.rm7_idx].unsqueeze(1).expand(B, self.output_dim)  # (B,PREDICT)

        yhat = torch.relu(base * (1.0 + boost * ratio))  # (B,PREDICT)
        return yhat
    
def clip_iqr(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    return np.clip(series, None, upper)

def compute_iqr_lower_bounds(train_df):
    lower_bounds = {}
    for menu, group in train_df.groupby('영업장명_메뉴명'):
        q1 = group['매출수량'].quantile(0.25)
        q3 = group['매출수량'].quantile(0.75)
        iqr = q3 - q1
        lower = max(q1 - 1.5 * iqr, 0)
        menu_key = menu[0] if isinstance(menu, tuple) else menu
        lower_bounds[menu_key] = lower
    return lower_bounds

from numpy.lib.stride_tricks import sliding_window_view

def train_lstm(train_df, use_validation=True, dropout=0.3):
    trained_models = {}

    for store_menu, group in tqdm(train_df.groupby(['영업장명_메뉴명'], sort=False), desc='Training LSTM'):
        # key를 문자열로 고정(파일명/딕셔너리 키 혼선 방지)
        key = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))

        store_train = group.sort_values('영업일자').copy()
        store_train['영업일자'] = pd.to_datetime(store_train['영업일자'])

        # 짧은 시계열 제외
        if len(store_train) < LOOKBACK + PREDICT + MIN_SEQUENCE_COUNT:
            continue

        # ---- 스플릿: 행 기준으로 먼저 나누고 train-part로만 scaler fit ----
        N = len(store_train)
        if use_validation:
            cutoff_row = max(LOOKBACK, int(round(N * 0.8)))
            cutoff_row = min(cutoff_row, N-1)
        else:
            cutoff_row = N

        scaler_xy    = MinMaxScaler()   # ['clipped_SQ','rolling_mean_7']를 "같이" fit/transform
        scaler_delta = MinMaxScaler()   # ['delta'] 전용

        # 1) train-part로 먼저 fit
        train_part = store_train.iloc[:cutoff_row].copy()
        _ = build_features(train_part, scaler_xy, scaler_delta, fit=True, date_col='영업일자')

        # 2) 같은 스케일러로 전체 transform (누수 없음)
        ft = build_features(store_train, scaler_xy, scaler_delta, fit=False, date_col='영업일자')
        display(ft[['holiday_prox_lead1','holiday_prox_lead4','holiday_prox_lead7']])
        # ---- 시퀀스화: 벡터화 (경고/속도 개선) ----
        vals = ft[FEATURES].values.astype(np.float32)   # (N, F)
        tgt  = ft['clipped_SQ'].values.astype(np.float32)
        wd   = ft['weekday'].values.astype(np.int64)
        ss   = ft['season'].values.astype(np.int64)
        mm   = ft['month_idx'].values.astype(np.int64)

        total_seq = len(ft) - LOOKBACK - PREDICT + 1
        if total_seq <= 0:
            continue

        # X: (S, LOOKBACK, F), y: (S, PREDICT), wd/ss: (S, LOOKBACK)
        X_np = sliding_window_view(vals, LOOKBACK, axis=0)[:total_seq]
        y_np = sliding_window_view(tgt, LOOKBACK + PREDICT, axis=0)[:total_seq, LOOKBACK:]
        wd_np = sliding_window_view(wd, LOOKBACK, axis=0)[:total_seq]
        ss_np = sliding_window_view(ss, LOOKBACK, axis=0)[:total_seq]
        mm_np = sliding_window_view(mm,   LOOKBACK, axis=0)[:total_seq]

        X = torch.from_numpy(X_np).float()
        X = ensure_time_major(X, LOOKBACK, len(FEATURES))
        y = torch.from_numpy(y_np).float()
        weekday_seqs = torch.from_numpy(wd_np).long()
        season_seqs  = torch.from_numpy(ss_np).long()
        month_seqs   = torch.from_numpy(mm_np).long()

        # ---- 시퀀스 기준 split ----
        if use_validation:
            split_idx = int(len(X) * 0.8)
            X_train, X_val = X[:split_idx], X[split_idx:]
            y_train, y_val = y[:split_idx], y[split_idx:]
            weekday_train, weekday_val = weekday_seqs[:split_idx], weekday_seqs[split_idx:]
            season_train,  season_val  = season_seqs[:split_idx],  season_seqs[split_idx:]
            month_train,   month_val   = month_seqs[:split_idx],   month_seqs[split_idx:]
        else:
            X_train, y_train = X, y
            weekday_train, season_train, month_train = weekday_seqs, season_seqs, month_seqs
            X_val = y_val = weekday_val = season_val = None

        X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)
        weekday_train, season_train , month_train = weekday_train.to(DEVICE), season_train.to(DEVICE) , month_train.to(DEVICE)
        if use_validation and X_val is not None:
            X_val, y_val = X_val.to(DEVICE), y_val.to(DEVICE)
            weekday_val, season_val, month_val = weekday_val.to(DEVICE), season_val.to(DEVICE), month_val.to(DEVICE)

        # ---- 모델 ----
        model = MultiEmbeddingLSTM(
            input_dim=len(FEATURES),
            hidden_dim=96,          # 96~128 
            num_layers=2,
            output_dim=PREDICT,
            use_month=True,
            dropout=dropout,
            proj_dim=32,            # 
            use_residual=False,
            head_mode="mul",          # ★
            r_scale_base=0.4,         # ★
            rm7_idx=FEAT['rolling_mean_7'],
            prox_idx=FEAT['holiday_prox'],
            mom7_idx=FEAT['momentum_7'],
            vol14_idx=FEAT['vol_14'],
            use_horizon_embed=True
        ).to(DEVICE)
        
        #loss function
        #criterion = nn.MSELoss()

        # 옵티마이저/스케줄러
        optimizer = torch.optim.AdamW(adamw_params(model, wd=1e-4), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5,
            patience=5,                 # 5   
            threshold=5e-4,             # 1e-3 -> 5e-4 (상대개선 0.05%)
            threshold_mode='rel',
            cooldown=1,                 # 0 -> 1 (한 번 내린 뒤 1에폭 쉬기)
            min_lr=3e-5                 # 1e-5 -> 3e-5 (너무 낮게 안내리기)
        )

        early = EarlyStopping(
            patience=PATIENCE, min_delta=0.005, mode='min',   # 0.5% 상대개선
            restore_best_weights=True, min_epochs=8, relative=True,
            smooth_beta=0.0  # 노이즈 크면 0.6~0.9로
        )

        train_losses = []
        train_losses_unw = []
        val_losses_unw = []
        val_smape_list = []

        for epoch in range(EPOCHS):
            model.train()
            total_loss_w, total_loss_unw, n_batches = 0.0, 0.0, 0
            last_grad_norm = 0.0
            idx = torch.randperm(len(X_train))

            # ★ anneal 스케줄 (초반 완만, 후반 강하게)
            lam = min(1.0, (epoch + 1) / 8.0)       # 8 에폭에 걸쳐 0->1
            alpha_t = 0.3 + 0.4 * lam               # sMAPE 비중: 0.3 -> 0.7

            for i in range(0, len(X_train), BATCH_SIZE):
                b = idx[i:i+BATCH_SIZE]
                xb = X_train[b]                    # (B,T,F)
                yb = y_train[b]
                wdb = weekday_train[b]
                ssb = season_train[b]
                mb = month_train[b]

                yhat = model(xb, wdb, ssb, mb)

                if ((epoch + 1) % 5 == 0) and (i == 0):
                    y0 = yhat[0].detach().float()             # (PREDICT,)
                    y0_np = y0.cpu().numpy()
                    # tqdm.write를 쓰면 진행바와 섞이지 않습니다.
                    tqdm.write(
                        f"[{key}] ep={epoch+1:02d} "
                        f"yhat[0]={np.round(y0_np, 4).tolist()} "
                        f"std={float(y0.std().item()):.4f}"
                    )
                    # print로 찍고 싶으면 flush도 추가
                    # print(..., flush=True)
                mom = xb[:, -1, FEAT['momentum_7']]
                prox= xb[:, -1, FEAT['holiday_prox']]
                vol = xb[:, -1, FEAT['vol_14']]
                # 타깃 기반 피크 가중치(현재 배치에서 큰 y 값에 더 큰 가중)
                peak_w = (yb.mean(dim=1) / (yb.mean() + 1e-6)).detach().clamp(0.8, 1.6)

                # 기존 피처기반 w와 결합
                w = (1.0 + 1.2*(F.relu(mom) + 0.6*prox + 0.4*F.relu(vol - 0.3))).clamp(1.0, 2.3)
                w = w * peak_w
                w = w.unsqueeze(1).expand_as(yhat)
                zero_w = (yb != 0).float()         # y=0이면 0, 아니면 1
                w = w * (0.1 + 0.9 * zero_w)       # 완전 제거가 불안하면 0.2 정도만 남기기

                # 마지막 시점의 이동평균을 기반으로 eps를 키움 (스케일 공간)
                eps_dyn = (0.25 * xb[:, -1, FEAT['rolling_mean_7']]).unsqueeze(1).expand_as(yhat)
                eps_dyn = eps_dyn.clamp(5e-4, 5e-2)  # 과도 방지

                # 가중/비가중 동시 계산
                loss_w   = weighted_huber_smape(yhat, yb, w=w,    delta=0.05, eps=eps_dyn, alpha=alpha_t)
                loss_unw = weighted_huber_smape(yhat, yb, w=None, delta=0.05, eps=eps_dyn, alpha=alpha_t)

                optimizer.zero_grad()
                loss_w.backward()
                total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ★ 추가
                #log
                # if (epoch+1) % 5 == 0 and i == 0:
                #     print(f"grad_norm(before_clip)≈{float(total_norm):.2f}")
                optimizer.step()
                total_loss_w   += loss_w.item()
                total_loss_unw += loss_unw.item()
                n_batches += 1

            train_losses.append(total_loss_w / max(1, n_batches))  
            if 'train_losses_unw' not in locals():
                train_losses_unw = []
            train_losses_unw.append(total_loss_unw / max(1, n_batches))

            if use_validation and X_val is not None and len(X_val) > 0:
                model.eval()
                with torch.no_grad():
                    val_out = model(X_val, weekday_val, season_val, month_val)
                    val_loss_unw = weighted_huber_smape(val_out, y_val, w=None, delta=0.05, eps=1e-3, alpha=alpha_t).item()
                    # ★ 대회 규칙: y==0 제외한 sMAPE(스케일 공간)
                    val_sm = smape_loss(val_out, y_val, eps=1e-3, reduction='mean', ignore_zero_target=True).item()


                val_losses_unw.append(val_loss_unw)
                
                #★ 역정규화 sMAPE (대회 지표)
                # 추가: sMAPE (scaled)
                val_smape_list.append(val_sm)


                # 스케줄러/얼리스탑
                scheduler.step(val_loss_unw)
                if early.step(val_loss_unw, model, epoch):
                    print(f"[{key}] Early stop @ {epoch+1} | best val_unw={min(val_losses_unw):.6f} | best_SMAPE={min(val_smape_list):.4f}")
                    break
            
        

        if use_validation:
            visualize_loss(
            train_losses, val_losses_unw, key,
            save=True, show=False, verbose=False,
            val_smape=val_smape_list,
        )
        else:
            visualize_loss(train_losses, None, key, save=True, show=False, verbose=False)

        # 동일 key로 저장
        trained_models[key] = {
            'model': model.eval(),
            'scaler_xy': scaler_xy,
            'scaler_delta': scaler_delta,
            'last_sequence': {
                'X': ft[FEATURES].values[-LOOKBACK:],
                'weekday': ft['weekday'].values[-LOOKBACK:],
                'season':  ft['season'].values[-LOOKBACK:],
                'month':   ft['month_idx'].values[-LOOKBACK:],
                'dates':   ft['영업일자'].values[-LOOKBACK:]
            },
            'last_obs_date': pd.to_datetime(ft['영업일자'].max())  # ★ 편의상 별도로도 저장
        }

    return trained_models


def visualize_loss(
    train_losses,
    val_losses,
    store_menu,
    save=False,
    out_dir="./loss_plots",
    show=False,
    verbose=False,
    val_smape=None      # sMAPE (scaled)
):
    import numpy as np
    plt.figure(figsize=(6,4))
    ax = plt.gca()  # 메인 축

    # 리스트/텐서 → float 배열 변환
    def to_float_array(xs):
        if xs is None:
            return np.array([], dtype=float)
        try:
            arr = np.asarray([float(x) for x in xs], dtype=float)
        except Exception:
            arr = np.array(xs, dtype=float)
        return arr

    tr = to_float_array(train_losses)
    va = to_float_array(val_losses)

    # 유한값만 마스크
    tr_mask = np.isfinite(tr)
    va_mask = np.isfinite(va)

    drew_any = False

    if tr.size > 0 and tr_mask.any():
        x_tr = np.arange(1, tr.size + 1)[tr_mask]
        y_tr = tr[tr_mask]
        ax.plot(x_tr, y_tr, marker='o', linewidth=1.5, label='Train Loss')
        drew_any = True

    if va.size > 0 and va_mask.any():
        x_va = np.arange(1, va.size + 1)[va_mask]
        y_va = va[va_mask]
        ax.plot(x_va, y_va, marker='o', linewidth=1.5, label='Validation Loss')
        drew_any = True

    title = f"[{store_menu}] Train vs Validation Loss"
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")

    # ===== sMAPE 보조축 =====
    has_smape = val_smape is not None and len(val_smape) > 0
    if has_smape:
        ax2 = ax.twinx()
        if has_smape:
            sm = to_float_array(val_smape)
            msk = np.isfinite(sm)
            if sm.size > 0 and msk.any():
                x_sm = np.arange(1, sm.size + 1)[msk]
                ax2.plot(x_sm, sm[msk], linestyle='--', marker='x', linewidth=1.2,
                         label='Val sMAPE (scaled)')
        ax2.set_ylabel("sMAPE")
        ax2.grid(False)

        # 범례 합치기
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    else:
        if drew_any:
            ax.legend(loc='upper right')

    if drew_any:
        ax.grid(True, alpha=0.4)
        ymin = min(np.min(tr[tr_mask]) if tr_mask.any() else np.inf,
                   np.min(va[va_mask]) if va_mask.any() else np.inf)
        ymax = max(np.max(tr[tr_mask]) if tr_mask.any() else -np.inf,
                   np.max(va[va_mask]) if va_mask.any() else -np.inf)
        if np.isfinite(ymin) and np.isfinite(ymax) and ymin != ymax:
            pad = 0.05 * (ymax - ymin)
            ax.set_ylim(ymin - pad, ymax + pad)
    else:
        msg = "No points to plot"
        if tr.size == 0 and (va is None or va.size == 0):
            msg += " (empty train/val lists)"
        elif (tr.size > 0 and not tr_mask.any()) and (va.size == 0 or not va_mask.any()):
            msg += " (all values are NaN/Inf)"
        ax.grid(True, alpha=0.4)
        ax.text(0.5, 0.5, msg, ha='center', va='center',
                transform=ax.transAxes, fontsize=12)

    # 파일명 안전 처리
    name_str = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
    safe_name = re.sub(r'[^\w\-_.]', '_', name_str)

    if verbose:
        print(f"[visualize_loss] {safe_name}: "
              f"len(train)={len(tr)}, finite(train)={tr_mask.sum()}, "
              f"len(val)={len(va)}, finite(val)={va_mask.sum()}, drew={drew_any}")

    if save:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"{safe_name}.png")
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
    if show and not save:
        plt.tight_layout()
        plt.show()

    plt.close()



def inverse_clipped_from_scaler(scaler_xy: MinMaxScaler, scaled_vals: np.ndarray) -> np.ndarray:
    """
    scaler_xy는 ['clipped_SQ','rolling_mean_7']에 대해 fit 되어 있음.
    clipped_SQ만 역변환하려면 2열 dummy를 만들어 1열만 복원.
    """
    dummy = np.zeros((len(scaled_vals), 2), dtype=np.float32)
    dummy[:, 0] = scaled_vals
    inv = scaler_xy.inverse_transform(dummy)[:, 0]
    return inv

#Prediction
def predict_lstm(
    test_df,
    trained_models,
    test_prefix: str,
    *,
    discontinued: dict[str, str | pd.Timestamp] | None = None,  # <- 추가
    rule: str = 'after',     # 'after' : 단종일자 이후(>) 0, 'on_or_after' : 단종일자 당일 포함(>=) 0
    grace_days: int = 0      # 유예일 (단종일자 + grace_days 이후부터 0)
):
    results = []

    # ----- (A) 단종 딕셔너리 Timestamp 정규화 -----
    cutoff_map = None
    if discontinued is not None:
        def _to_ts(v):
            return v if isinstance(v, pd.Timestamp) else pd.to_datetime(v)
        cutoff_map = {k: _to_ts(v) for k, v in discontinued.items()}
        if grace_days != 0:
            for k in cutoff_map:
                cutoff_map[k] = cutoff_map[k] + pd.Timedelta(days=grace_days)


    for store_menu, store_test in test_df.groupby(['영업장명_메뉴명'], sort=False):
        key = store_menu if isinstance(store_menu, str) else "_".join(map(str, store_menu))
        if key not in trained_models:
            continue

        model        = trained_models[key]['model']
        scaler_xy    = trained_models[key]['scaler_xy']
        scaler_delta = trained_models[key]['scaler_delta']

        store_test_sorted = store_test.sort_values('영업일자').copy()
        store_test_sorted['영업일자'] = pd.to_datetime(store_test_sorted['영업일자'])

        # 학습과 동일 전처리 (fit=False)
        ft = build_features(store_test_sorted, scaler_xy, scaler_delta, fit=False, date_col='영업일자')

        # 최근 LOOKBACK 시퀀스 확보
        if len(ft) < LOOKBACK:
            last_seq   = trained_models[key]['last_sequence']
            x_input    = torch.tensor([last_seq['X']]).float().to(DEVICE)
            x_input = ensure_time_major(x_input, LOOKBACK, len(FEATURES))  # ★
            weekday_seq= torch.tensor([last_seq['weekday']]).long().to(DEVICE)
            season_seq = torch.tensor([last_seq['season']]).long().to(DEVICE)
            month_seq  = torch.tensor([last_seq['month']]).long().to(DEVICE)
            last_obs_date = pd.to_datetime(trained_models[key].get('last_obs_date', last_seq['dates'][-1]))
        else:
            recent     = ft.iloc[-LOOKBACK:]
            x_input    = torch.tensor([recent[FEATURES].values]).float().to(DEVICE)
            x_input = ensure_time_major(x_input, LOOKBACK, len(FEATURES))  # ★
            weekday_seq= torch.tensor([recent['weekday'].values]).long().to(DEVICE)
            season_seq = torch.tensor([recent['season'].values]).long().to(DEVICE)
            month_seq  = torch.tensor([recent['month_idx'].values]).long().to(DEVICE)
            last_obs_date = pd.to_datetime(recent['영업일자'].max())

        # 예측 (스케일 공간)
        with torch.no_grad():
            pred_scaled = model(x_input, weekday_seq, season_seq, month_seq).squeeze().cpu().numpy()

        # 역정규화 + 하한 클리핑(메뉴별 하한 없으면 1)
        restored = inverse_clipped_from_scaler(scaler_xy, pred_scaled)
        restored = np.maximum(restored, 1.0)

        # ----- (B) 단종 처리: 예측 구간 실제 날짜와 비교 -----
        # 예측 구간의 "실제 달력 날짜" 생성 (last_obs_date 다음날부터 PREDICT일)
        horizon_dates = pd.date_range(start=last_obs_date + pd.Timedelta(days=1),
                                      periods=PREDICT, freq='D')

        if cutoff_map is not None:
            cutoff = cutoff_map.get(key, None)
            if cutoff is not None:
                if rule == 'on_or_after':
                    zero_mask = horizon_dates >= cutoff
                else:  # 'after'
                    zero_mask = horizon_dates > cutoff
                restored = np.where(zero_mask, 0.0, restored)

        # 제출 포맷
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
        for d, val in zip(pred_dates, restored):
            results.append({
                '영업일자': d,
                '영업장명_메뉴명': key,      # ★ 전체 문자열로 저장
                '매출수량': float(val)
            })

    return pd.DataFrame(results)

def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    # (영업일자, 메뉴) → 매출수량 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:  # 메뉴명들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df


In [68]:
#Data load
train = pd.read_csv('./train/train.csv')
train = generate_combined_holiday_list(train, solar_md_holidays, lunar_solar_dates)
train = filter_all_menus_by_leading_zeros(
    train,
    min_zero_days=90,
    apply_to_stores=['담하','라그로타','미라시아' ]  # 여기에 대상 업장명만 나열
)
trained_models = train_lstm(train, use_validation=True, dropout=0.1)

Training LSTM:   0%|          | 0/193 [00:00<?, ?it/s]

,holiday_prox_lead1,holiday_prox_lead4,holiday_prox_lead7
0,0.9,0.6,0.3
1,0.8,0.5,0.2
2,0.7,0.4,0.1
3,0.6,0.3,0.0
4,0.5,0.2,0.0
...,...,...,...
527,0.4,0.1,0.0
528,0.3,0.0,0.0
529,0.2,0.0,0.0
530,0.1,0.0,0.0


Training LSTM:   0%|          | 0/193 [00:02<?, ?it/s]

[느티나무 셀프BBQ_1인 수저세트] ep=05 yhat[0]=[0.9821000099182129, 0.9775000214576721, 0.9991999864578247, 0.9311000108718872, 0.9937000274658203, 0.9334999918937683, 0.986299991607666] std=0.0280


Training LSTM:   0%|          | 0/193 [00:03<?, ?it/s]

...

,holiday_prox_lead1,holiday_prox_lead4,holiday_prox_lead7
2660,0.9,0.6,0.3
2661,0.8,0.5,0.2
2662,0.7,0.4,0.1
2663,0.6,0.3,0.0
2664,0.5,0.2,0.0
...,...,...,...
3187,0.4,0.1,0.0
3188,0.3,0.0,0.0
3189,0.2,0.0,0.0
3190,0.1,0.0,0.0


Training LSTM:   3%|▎         | 5/193 [00:33<20:17,  6.48s/it]

[느티나무 셀프BBQ_본삼겹 (단품,실내)] ep=05 yhat[0]=[0.39739999175071716, 0.39739999175071716, 0.39730000495910645, 0.39730000495910645, 0.3971000015735626, 0.39730000495910645, 0.39719998836517334] std=0.0001


Training LSTM:   3%|▎         | 5/193 [00:34<21:24,  6.83s/it]


KeyboardInterrupt: 

In [29]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))
df = pd.read_csv('./train/train.csv')
lower_bound_dict = compute_iqr_lower_bounds(df)
for path in test_files:
    test_df = pd.read_csv(path)
    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_lstm(test_df, trained_models, test_prefix,
        discontinued=DISCONTINUED,
        rule='after',      # 단종일 "이후" 0
        grace_days=0       # 유예일 없으면 0
    )
    all_preds.append(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)

/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_13756/4029812827.py:1117: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  x_input    = torch.tensor([recent[FEATURES].values]).float().to(DEVICE)


In [30]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('./Prediction/model_v7_6.csv', index=False, encoding='utf-8-sig')
result = pd.read_csv('./Prediction/model_v7_6.csv')
display(result.head())

/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_13756/4029812827.py:1169: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '15.837484359741211' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_13756/4029812827.py:1169: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '53.695152282714844' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
/var/folders/r4/sdnz117n6pl22zr9jhhv5vbm0000gn/T/ipykernel_13756/4029812827.py:1169: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12.29704475402832' has

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,15.837484,53.695152,12.297045,7.368956,1.803117,2.752515,19.828720,4.089544,3.844003,...,24.991623,47.579887,49.869877,28.451027,199.355865,54.717335,37.135948,92.756462,22.190117,35.140022
1,TEST_00+2일,15.831112,53.511070,12.296339,7.369354,1.803349,2.693405,19.829111,4.062624,3.848145,...,25.021578,47.539658,49.770428,28.419905,199.089371,54.769630,37.141342,92.784309,22.182978,34.993832
2,TEST_00+3일,15.834917,53.443050,12.299439,7.370805,1.803117,2.799441,19.827637,4.214483,3.839733,...,25.008457,47.504742,49.846344,28.401894,198.689056,54.856995,37.180283,92.740921,22.166519,34.945156
3,TEST_00+4일,15.829988,53.362171,12.297628,7.370432,1.802881,2.793416,19.827194,4.233429,3.837599,...,24.996315,47.517807,49.762733,28.416643,198.815811,54.992432,37.149620,92.761505,22.156418,34.925468
4,TEST_00+5일,15.824183,53.536663,12.296406,7.370214,1.803172,2.660748,19.829220,4.134252,3.834826,...,24.992456,47.498653,49.774269,28.399210,198.662704,55.140171,37.140465,92.709641,22.171331,34.901272
